# 06: Puzzles and logic: invariants, counterexamples and LLM sense

**Practice set:** `practice/06-puzzles-logic.md` · **Exam tasks:** Jan 18, Section A1: Cracking the Imperial Glyphs, Filtering Fakes with Faulty Oracle, Smart Student Search, Pikachu's Path to Victory · **Moves:** M8 (parity / colouring), M10 (kill options with counterexamples), M14 (LLM reality check), M17 (align and decode), M1 (read the definition, not your memory)

## Hook

Your robot vacuum has to clean a 5 × 5 tiled floor. It moves one tile up, down, left or right, it never crosses its own track, and it starts on the tile right next to its charging dock. You write a planner that searches for a route covering *every* tile and ending on the dock. It runs and runs and never finds one. Is your planner buggy, or is a full route impossible? And how would you *know*, without waiting for the search to finish?

Meanwhile, the language model in your VLA stack (the part that turns "pick up the strawberry" into a plan) tells you, with total confidence, that "strawberry" has two r's.

Both stories are about the same skill: **knowing what is possible before you search or trust.** A one-line argument can put a ceiling on any route your planner could ever find. A one-line fact about how language models read text tells you which questions they will get wrong. This notebook builds both, plus the other moves the exam's puzzle section keeps using: tracing code by hand, "if P then Q" logic, finding a hidden rule, designing a formula from rules, and decoding an unknown language.

## Before you start

- **You need:** Python (loops, functions, sets, dicts) and the idea of even and odd. Everything else is defined when it first appears.
- **You'll be able to:** put a ceiling on a path with a colouring, confirm it with a search, trace nested loops without slips, decide which cases can break an "if ... then ..." rule, find a hidden rule by killing wrong guesses, check formulas against design rules, compute edit distance and Jaro similarity by hand, decode a small invented language, and predict where an LLM will fail.
- **How to use this notebook:** run the cells in order. When you see **PREDICT**, write your guess *before* running the next cell. When you see **BUILD**, write the function yourself; the test cell tells you when it's right. Hints sit in collapsed blocks. Open one only after a real attempt.
- **Pen and paper stay on the desk.** The exam has no computer. The notebook is for checking and exploring *after* you've reasoned it out.

In [ ]:
import prep
import math
import random
from collections import Counter
from itertools import permutations
import numpy as np
import matplotlib.pyplot as plt

ORTH = [(1, 0), (-1, 0), (0, 1), (0, -1)]                                    # up, down, left, right
KNIGHT = [(1, 2), (2, 1), (-1, 2), (-2, 1), (1, -2), (2, -1), (-1, -2), (-2, -1)]

---
## 1. Colour the floor: an invariant

**Idea.** An **invariant** is something a move cannot change, or changes in a completely predictable way. If you can find one, you can rule things out without trying them.

Here is the invariant for grids. Label each cell (row, column), with (0, 0) in the top-left. Colour it like a chessboard:
- **dark (D)** if row + column is **even** (0, 2, 4, ...),
- **light (L)** if row + column is **odd** (1, 3, 5, ...).

Whether a number is even or odd is called its **parity**.

**Smallest example.** A 3 × 3 board, and one path on it. Watch the last column.

In [ ]:
def colour(cell):
    return "D" if (cell[0] + cell[1]) % 2 == 0 else "L"

n = 3
for r in range(n):
    print("  ".join(colour((r, c)) for c in range(n)))
print()

path = [(0, 0), (0, 1), (0, 2), (1, 2), (1, 1), (1, 0), (2, 0)]
rows = [[k, cell, cell[0] + cell[1], colour(cell)] for k, cell in enumerate(path)]
prep.table(rows, header=["step", "cell", "row + col", "colour"])

**PREDICT:** look only at the colour column. What pattern do you see? Will it happen on *every* path made of up/down/left/right moves, or did we get lucky? Write your answer and a one-line reason.

<details><summary>Check your reasoning</summary>

The colours alternate: D, L, D, L, ... Every move changes the row **or** the column by exactly 1, so row + column changes by exactly 1. Adding or subtracting 1 always flips parity, so the colour flips on **every** move, on every path, on every board. That's the invariant: *a path's colours alternate.*
</details>

**PREDICT:** on the 3 × 3 board, a path starts at (0, 0) and must end at (0, 1). It never revisits a cell. Count the D cells and the L cells on the board. What is the **largest** number of cells such a path could possibly have? Write a number and your reason before opening the block.

<details><summary>Check your reasoning</summary>

The start (0, 0) is D and the end (0, 1) is L. The colours alternate, so the path looks like D, L, D, L, ..., L: it has **exactly as many L cells as D cells**. The board has 5 D cells but only 4 L cells, so the path can use at most 4 L cells, and therefore at most 4 D cells: **at most 8 cells.** One D cell must always be left out, no matter how clever the path.
</details>

**The aha: a colouring gives you a ceiling before you search.** Counting colours tells you the most any path *could* do. It doesn't yet tell you that some path *does* reach that number. That's the job of section 2.

### BUILD 1: `colour_counts`
Return `(dark, light)`: how many cells of an n × n board have row + column even, and how many have it odd.

In [ ]:
def colour_counts(n):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(colour_counts, [
    ((1,), (1, 0)),
    ((2,), (2, 2)),
    ((3,), (5, 4)),
    ((4,), (8, 8)),
])

### Knights change colour too

A chess **knight** moves two squares in one direction and one square at right angles to it, like an L. The list `KNIGHT` at the top holds its 8 moves as (change in row, change in column).

**PREDICT:** does a knight move always flip the colour, never flip it, or sometimes? Think about what each move does to row + column, then run the cell.

In [ ]:
rows = [[m, m[0] + m[1], "odd" if (m[0] + m[1]) % 2 else "even"] for m in KNIGHT]
prep.table(rows, header=["move (dr, dc)", "change in row + col", "parity of the change"])

Every knight move changes row + column by ±1 or ±3, an odd number, so **every knight move flips the colour**, just like a one-step move. The same ceiling argument works for knights. (Careful: the reverse is false. "The colour flipped" does *not* mean "it was a knight move", because one-step moves flip it too. Remember this in section 4.)

---
## 2. Search: try everything, but cleverly

**Idea.** A ceiling says "no path can beat N". To know the true maximum you also need a path that *reaches* the ceiling, or proof that none does. For small boards, a computer can simply try every path.

**Smallest example: brute force.** On a 2 × 2 board, from (0, 0) to (0, 1): list every order in which you could visit cells, then throw away the orders that break a move rule.

In [ ]:
def follows_moves(seq, moves):
    return all((b[0] - a[0], b[1] - a[1]) in moves for a, b in zip(seq, seq[1:]))

start, end = (0, 0), (0, 1)
middle = [(1, 0), (1, 1)]                     # cells that could sit between start and end
rows = []
for k in range(len(middle) + 1):
    for mid in permutations(middle, k):
        seq = [start, *mid, end]
        rows.append([" -> ".join(map(str, seq)), len(seq), "valid" if follows_moves(seq, ORTH) else "breaks a move rule"])
prep.table(rows, header=["candidate order", "cells", "verdict"])

**PREDICT:** on the 5 × 5 board there are 23 cells that could sit between the start and the end. Roughly how many candidate orders would brute force have to check? Thousands? Billions? More? Guess, then run.

In [ ]:
total = sum(math.perm(23, k) for k in range(24))    # perm(23, k) = orders of k cells chosen from 23
print(f"{total:.2e} candidate orders")
print(f"at a billion checks per second: {total / 1e9 / 3600 / 24 / 365:.1e} years")

Brute force is hopeless. Almost all of those orders break a move rule at the very first step, though, so a smarter search never builds them.

**Depth-first search (DFS) with backtracking** builds a path one cell at a time:
1. From the current cell, try each move in turn.
2. If the move lands on the board and on an unvisited cell, step there and repeat from step 1. This is going *deep* first.
3. If you are on the end cell, record the path's length and stop extending it. A path must *finish* on the end cell, and it can't pass through and come back.
4. When every move from a cell has been tried, **backtrack**: undo the last step (un-visit the cell) and try the next move one level up.

Here is the whole search on the 2 × 2 board:

```
(0,0)
├── (1,0)
│   └── (1,1)
│       └── (0,1)   end reached: 4 cells   <- best
└── (0,1)           end reached: 2 cells
```

The natural way to write this is **recursion**: a function that calls itself on a smaller job. `dfs(cell, visited)` handles "continue the path from `cell`" and calls `dfs` on each legal next cell. A tiny example of recursion, a countdown:

```python
def down(k):
    print(k)
    if k > 0:
        down(k - 1)     # the same job, one step smaller
```

### BUILD 2: `longest_path`
Return the largest number of cells (counting start and end) on any path from `start` to `end` on an n × n board that uses only steps from `moves` and never revisits a cell. Return 0 if `end` can't be reached at all.

In [ ]:
def longest_path(n, start, end, moves):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(longest_path, [
    ((2, (0, 0), (0, 1), ORTH), 4),
    ((3, (0, 0), (2, 2), ORTH), 9),       # a snake: row 0 left to right, row 1 right to left, row 2 left to right
    ((3, (0, 0), (2, 2), KNIGHT), 5),
    ((3, (0, 0), (1, 1), KNIGHT), 0),     # can a knight ever reach the centre of a 3 x 3 board?
])

### EXPLORE: ceiling vs search

The cell below computes the colour ceiling for several boards (the reasoning from section 1, written as code) and compares it with your search.

**PREDICT** for each row: will the search reach the ceiling? Pay special attention to the 3 × 3 knight board, and remember the last test above.

In [ ]:
def colour_ceiling(n, start, end):
    dark, light = colour_counts(n)
    count = {"D": dark, "L": light}
    same, other = colour(start), "L" if colour(start) == "D" else "D"
    if colour(end) == same:      # D, L, ..., D: one more of the start colour than the other
        return min(2 * count[same] - 1, 2 * count[other] + 1)
    return 2 * min(count[same], count[other])   # D, L, ..., L: equal numbers of each

if prep.ready(colour_counts, 2) and prep.ready(longest_path, 2, (0, 0), (0, 1), ORTH):
    boards = [(3, (0, 0), (2, 2), ORTH, "step"), (4, (0, 0), (3, 3), ORTH, "step"), (4, (0, 0), (1, 1), ORTH, "step"),
              (3, (0, 0), (2, 2), KNIGHT, "knight"), (4, (0, 0), (3, 3), KNIGHT, "knight")]
    rows = []
    for n_, s, e, mv, name in boards:
        rows.append([f"{n_}x{n_}", name, s, e, colour_ceiling(n_, s, e), longest_path(n_, s, e, mv)])
    prep.table(rows, header=["board", "moves", "start", "end", "colour ceiling", "search finds"])

**PREDICT:** the 3 × 3 knight row looks strange. Draw every knight move on a 3 × 3 board on paper (a line between two cells if a knight can jump between them). What shape do you get? Then run the cell.

In [ ]:
plt.figure(figsize=(3.5, 3.5))
for r in range(3):
    for c in range(3):
        for dr, dc in KNIGHT:
            r2, c2 = r + dr, c + dc
            if 0 <= r2 < 3 and 0 <= c2 < 3:
                plt.plot([c, c2], [r, r2], color="tab:blue", alpha=0.6)
        plt.scatter(c, r, s=300, color="0.25" if colour((r, c)) == "D" else "0.85", edgecolor="k", zorder=3)
plt.gca().invert_yaxis(); plt.xticks(range(3)); plt.yticks(range(3)); plt.xlabel("column"); plt.ylabel("row")
plt.title("Knight moves on 3 x 3"); plt.show()

**The struggle, resolved.** On the 3 × 3 knight board the ceiling is **not** reached. The knight moves form a single loop through the 8 outer cells, and the centre has no moves at all. From (0, 0) to (2, 2) the knight can only go one way or the other way round that loop, so the true answer is far below the colour count. A ceiling is a promise that you can't do *better*. It is never a promise that you can do *that well*. A full answer needs both halves: **a ceiling, and a path that reaches it.**

### EXPLORE: pruning

**Pruning** means cutting off branches of the search that can't lead to a better answer. The simplest prune: once you've found a path that reaches the ceiling, stop, because nothing can beat it. The cell counts how many times `dfs` gets called with and without that prune.

**PREDICT:** on which board will the prune save nothing at all?

In [ ]:
def count_calls(n, start, end, moves, ceiling=None):
    best, calls = 0, 0
    def dfs(cell, visited):
        nonlocal best, calls
        calls += 1
        if cell == end:
            best = max(best, len(visited)); return
        for dr, dc in moves:
            if ceiling is not None and best == ceiling:
                return                                   # the prune: nothing can beat the ceiling
            nxt = (cell[0] + dr, cell[1] + dc)
            if 0 <= nxt[0] < n and 0 <= nxt[1] < n and nxt not in visited:
                visited.add(nxt); dfs(nxt, visited); visited.remove(nxt)
    dfs(start, {start})
    return calls

if prep.ready(colour_counts, 2):
    rows = []
    for n_, s, e, mv, name in [(4, (0, 0), (3, 3), ORTH, "step"), (4, (0, 0), (3, 3), KNIGHT, "knight"),
                               (3, (0, 0), (2, 2), KNIGHT, "knight")]:
        rows.append([f"{n_}x{n_}", name, count_calls(n_, s, e, mv), count_calls(n_, s, e, mv, colour_ceiling(n_, s, e))])
    prep.table(rows, header=["board", "moves", "calls, no prune", "calls, stop at ceiling"])

When the ceiling is never reached, the prune never fires, so the search must explore everything to be sure. That's another reason the exam favours the colouring argument: on paper you can't run a million calls.

**Now do practice problem 6.1 on paper.** Find the ceiling first, then draw a path that reaches it. Type your path into the cell below to check that it's legal (the checker never tells you the maximum), then check your answer.

In [ ]:
my_path = [(0, 0), ..., (0, 1)]   # your path for 6.1, as a list of (row, col) cells

def check_path(path, n, start, end, moves):
    if any(c is ... for c in path):
        print("Fill in your path first (replace the ...)."); return
    problems = []
    if path[0] != start: problems.append(f"it should start at {start}")
    if path[-1] != end: problems.append(f"it should end at {end}")
    if len(set(path)) != len(path): problems.append("it visits some cell twice")
    off = [c for c in path if not (0 <= c[0] < n and 0 <= c[1] < n)]
    if off: problems.append(f"{off[0]} is off the board")
    bad = [(a, b) for a, b in zip(path, path[1:]) if (b[0] - a[0], b[1] - a[1]) not in moves]
    if bad: problems.append(f"{bad[0][0]} -> {bad[0][1]} is not an allowed move")
    print("Not a legal path: " + "; ".join(problems) if problems else f"Legal path with {len(path)} cells.")

check_path(my_path, 5, (0, 0), (0, 1), ORTH)

In [ ]:
prep.check("6.1", ...)   # the maximum number of cells

---
## 3. Tracing code by hand

**Idea.** A **trace table** has one column per variable and one row each time the innermost line runs. You write every intermediate value, so a slip shows up in one row instead of hiding in the final number.

**Convention (just learn it):** `range(a, b)` gives a, a+1, ..., b−1. It **stops before b**. So `range(1, 4)` is 1, 2, 3, and `range(i, 4)` starts at whatever `i` currently is.

**Smallest example.**
```python
x = 0
for i in range(1, 4):
    for j in range(i, 4):
        if (i + j) % 2 == 0:
            x += i * j
        else:
            x -= 1
```

**PREDICT:** before running anything, (a) how many rows will the trace table have? (b) what is the final x? Do it on paper, with the columns i, j, i + j, (i + j) % 2, action, x.

In [ ]:
rows = []
x = 0
for i in range(1, 4):
    for j in range(i, 4):
        if (i + j) % 2 == 0:
            x += i * j
            action = f"x += {i * j}"
        else:
            x -= 1
            action = "x -= 1"
        rows.append([i, j, i + j, (i + j) % 2, action, x])
prep.table(rows, header=["i", "j", "i+j", "(i+j)%2", "action", "x after"])

**Check a second way.** The final x is (sum of i·j over the rows where the test passes) − (number of rows where it fails). Computing it this way uses different arithmetic, so a slip in one method won't repeat in the other.

In [ ]:
hits = [(i, j) for i in range(1, 4) for j in range(i, 4) if (i + j) % 2 == 0]
misses = [(i, j) for i in range(1, 4) for j in range(i, 4) if (i + j) % 2 != 0]
print("hits:", hits, " sum of i*j =", sum(i * j for i, j in hits))
print("misses:", misses, " count =", len(misses))
print("x =", sum(i * j for i, j in hits) - len(misses))

### EXPLORE: the classic slip
A common misread is to start the inner loop at 1 instead of at i. **PREDICT:** if the inner loop were `range(1, 4)`, how many rows would there be, and what would x be? Then run.

In [ ]:
x = 0
count = 0
for i in range(1, 4):
    for j in range(1, 4):
        x += i * j if (i + j) % 2 == 0 else -1
        count += 1
print("rows:", count, " final x:", x)

**Now trace practice problem 6.2 by hand** with a full table, check it a second way, and only then check the answer. Don't run the code from the problem.

In [ ]:
prep.check("6.2", ...)   # the printed value

---
## 4. "If P then Q": what can break it?

**Idea.** "If P then Q" is a promise about the cases where P happens. It says **nothing** about cases where P doesn't happen.

**Smallest example.** A rule for your gripper: *"If the gripper is closed (P), then the object sensor reads 'holding' (Q)."* There are only four possible situations. A table that lists every combination of true/false is called a **truth table**.

**PREDICT:** which of the four rows below **break** the rule? Decide for each row before running the next cell.

| gripper closed (P) | sensor says holding (Q) |
|---|---|
| yes | yes |
| yes | no |
| no | yes |
| no | no |

In [ ]:
rows = []
for P in (True, False):
    for Q in (True, False):
        rows.append([P, Q, "BROKEN" if P and not Q else "kept"])
prep.table(rows, header=["gripper closed (P)", "says holding (Q)", "rule"])

Only **P true and Q false** breaks the rule. When the gripper is open (P false), the rule has made no promise, so it can't be broken, whatever the sensor says. People often feel that row 3 (open gripper, sensor says holding) breaks the rule. It doesn't: maybe the sensor is faulty, but this rule says nothing about open grippers.

Two traps:
- **The converse.** "If P then Q" is not the same as "if Q then P". Section 1 had an example: "if it's a knight move, the colour flips" is true, but "if the colour flips, it was a knight move" is false.
- **"If" vs "if and only if".** "P if and only if Q" makes *both* promises. A plain "if" makes only one.

### Cards with a hidden side
Each card has a letter on one side and a number on the other. The rule: *"If a card shows a vowel, then its other side is an even number."* You see four cards: **E**, **K**, **4**, **7**.

**PREDICT:** which cards must you turn over to check the rule? Pick the smallest set that is sure to catch any violation. Write it down first.

<details><summary>Hint</summary>

Don't ask "which card could *confirm* the rule?" Ask, for each card: "is there anything on the hidden side that would make this card a **P-and-not-Q** case?" If no hidden value could break the rule, turning that card over is wasted effort.
</details>

### BUILD 3: `must_flip`
`cards` is a dict from card name to `(side, value)`:
- `side` is `"P"` if the visible side tells you about P, or `"Q"` if it tells you about Q;
- `value` is `True` or `False`: whether that statement holds on the visible side.

For the vowel rule, card E is `("P", True)` (it is a vowel) and card 7 is `("Q", False)` (it is not even). Return the list of card names that must be flipped, in the order they appear in `cards`. Work it out by trying both possible hidden values for each card.

In [ ]:
def must_flip(cards):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(must_flip, [
    (({"E": ("P", True), "K": ("P", False), "4": ("Q", True), "7": ("Q", False)},), ["E", "7"]),
    (({"x": ("Q", True)},), []),
    (({"a": ("P", True), "b": ("Q", False)},), ["a", "b"]),
])

**Now do practice problem 6.4 on paper** (write each card as P/Q and true/false first), then check. Answer with letters, e.g. `"A,B"`.

In [ ]:
prep.check("6.4", ...)

---
## 5. Finding a hidden rule: guess, then try to kill the guess

**Idea.** A **hypothesis** is a guess at the rule. A **counterexample** is an example where the guess gives the wrong output. One counterexample kills a hypothesis. No number of agreeing examples can prove one. So the method is:
1. write down several hypotheses, including silly-looking ones;
2. test **every** hypothesis on **every** example;
3. when they all die, ask what the killing example has that the others don't, and change the *kind* of feature you look at (single symbols, positions, neighbours, length, ...).

**Smallest example.** A hidden function turns words into numbers. Four hypotheses, tested one example at a time:

In [ ]:
examples = [("cat", 1), ("robot", 2), ("sensor", 2), ("arm", 1), ("camera", 3)]
hypotheses = {
    "length - 2": lambda w: len(w) - 2,
    "length // 2": lambda w: len(w) // 2,
    "number of vowels": lambda w: sum(ch in "aeiou" for ch in w),
    "number of o's": lambda w: w.count("o"),
}
alive = set(hypotheses)
rows = []
for word, out in examples:
    marks = []
    for name, h in hypotheses.items():
        ok = h(word) == out
        marks.append(f"{h(word)} {'ok' if ok else 'X'}")
        if not ok:
            alive.discard(name)
    rows.append([word, out, *marks, ", ".join(sorted(alive))])
prep.table(rows, header=["example", "f", *hypotheses, "still alive"])

Look at "length // 2": it fits the first two examples perfectly and dies on the third. If you'd stopped after two examples you'd have the wrong rule. "Number of vowels" survives all five. That doesn't *prove* it, but it's the only candidate left.

**PREDICT** (then think, no cell to run): suppose a sixth example were ("sky", 1). Which hypothesis survives now? If none, what would you try next? (Step 3 of the method: what does "sky" have that the others don't?)

**Now do practice problem 6.6 on paper.** Write at least three hypotheses and test each on all five examples.

In [ ]:
prep.check("6.6", ...)

### BUILD 4: your 6.6 rule as code (only after 6.6 says correct)

Did you try "count the 1s"? It dies on 1111. "Count the 0s" is sneakier: it fits 0110 and 1111 and first fails on 1010, which is section 5's "length // 2" lesson again. The rule that fits all five examples counts the places where a bit differs from its neighbour. Write it. The tests are the five examples from the problem.

In [ ]:
def adjacent_changes(bits):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(adjacent_changes, [
    (("0110",), 2), (("1111",), 0), (("1010",), 3), (("0001",), 1), (("1001",), 2),
])

### EXPLORE: the colouring argument returns
**PREDICT:** is `adjacent_changes(bits)` even exactly when the first and last bits are equal? Guess, then test it on every bit string of length 1 to 12.

In [ ]:
if prep.ready(adjacent_changes, "01"):
    agree = total = 0
    for L in range(1, 13):
        for k in range(2 ** L):
            bits = format(k, f"0{L}b")
            total += 1
            agree += (adjacent_changes(bits) % 2 == 0) == (bits[0] == bits[-1])
    print(f"the rule held on {agree} of {total} strings")

**Why:** every change flips the bit, exactly as every move flips the colour in section 1. After an even number of flips you are back where you started.

**The Week 4 connection.** Convolving with the kernel [1, −1] gives the differences d = x[k+1] − x[k]. Each difference is +1 (a jump up), −1 (a jump down) or 0. Adding the **absolute values** counts the jumps, which is this rule. Adding the differences *themselves* telescopes: everything cancels except last − first.

In [ ]:
bits = "0011101"
x = [int(b) for b in bits]
d = [x[k + 1] - x[k] for k in range(len(x) - 1)]
print("bits:", x)
print("differences [1,-1]:", d)
print("sum of |d| =", sum(abs(v) for v in d), "  sum of d =", sum(d), "  last - first =", x[-1] - x[0])

---
## 6. Designing a formula from rules: the rules × options grid

**Idea.** Some exam questions give you rules and several candidate formulas: "which satisfy all the rules?" Don't try to prove the right one first. Draw a grid with **options as rows and rules as columns**, and hunt for a **counterexample** in each cell. One counterexample kills the whole row.

Where counterexamples hide:
- **extreme inputs:** zeros, equal inputs, a very large value, the edge of the allowed range;
- **swapped inputs** (for symmetry rules);
- **both directions of an "if and only if"** (it makes two promises; check each);
- inputs that **divide by zero** or make something negative.

Words you need:
- **symmetric:** S(a, b) = S(b, a) for all a, b. Example: |a − b| is symmetric, a − b is not.
- **strictly decreasing:** a bigger input gives a strictly smaller output; "stays the same" is not allowed.
- **exp:** e ≈ 2.718, and exp(x) = eˣ. It is always positive, e⁰ = 1, and it grows as x grows. So for x ≥ 0, e^(−x) lies in (0, 1], and equals 1 only at x = 0.

**Smallest example.** A similarity score S(a, b) for two battery levels a, b between 0 and 100. Rules: **R1** S = 1 if and only if a = b. **R2** S is symmetric. **R3** 0 ≤ S ≤ 1. Here is the candidate S = 1 − (a − b)/100 checked at five test points, every step printed:

In [ ]:
def S_Q(a, b):
    return 1 - (a - b) / 100

points = [(30, 70), (50, 50), (0, 0), (0, 100), (100, 0)]
rows = []
for a, b in points:
    v, v_swap = S_Q(a, b), S_Q(b, a)
    r1 = (abs(v - 1) < 1e-12) == (a == b)
    rows.append([(a, b), round(v, 3), round(v_swap, 3), "ok" if r1 else "X", "ok" if abs(v - v_swap) < 1e-12 else "X",
                 "ok" if 0 <= v <= 1 else "X"])
prep.table(rows, header=["(a, b)", "S(a,b)", "S(b,a)", "R1", "R2", "R3"])

One row with an X is enough: this option is dead on R2 and R3. Here are the other candidates:

| option | S(a, b) |
|---|---|
| P | 1 − \|a − b\| / 100 |
| Q | 1 − (a − b) / 100 |
| R | min(a, b) / max(a, b) |
| S | e^(−\|a − b\|) |
| U | a / b |

**PREDICT:** fill in the grid on paper (5 options × 3 rules). For each X, write the test point that kills it. Which options survive?

In [ ]:
battery_options = {
    "P": lambda a, b: 1 - abs(a - b) / 100,
    "Q": lambda a, b: 1 - (a - b) / 100,
    "R": lambda a, b: min(a, b) / max(a, b),
    "S": lambda a, b: math.exp(-abs(a - b)),
    "U": lambda a, b: a / b,
}
battery_rules = {
    "R1 (=1 iff a=b)": lambda f, p: (abs(f(*p) - 1) < 1e-12) == (p[0] == p[1]),
    "R2 symmetric": lambda f, p: abs(f(*p) - f(p[1], p[0])) < 1e-12,
    "R3 in [0,1]": lambda f, p: 0 <= f(*p) <= 1,
}

### BUILD 5: `rule_grid`
- `options`: dict, name → function.
- `rules`: dict, name → a checker `rule(f, point)` that returns True if option `f` obeys the rule at `point`.
- `test_points`: a list of points (tuples). Call an option as `f(*point)` inside your checkers.

Return a dict of dicts: `grid[option][rule]` is the **first** test point where the rule fails, or `None` if it never fails. If a checker raises an error (for example, division by zero), count that as a failure at that point.

In [ ]:
def rule_grid(options, rules, test_points):
    # your code here
    raise NotImplementedError

In [ ]:
toy_opts = {"double": lambda x: 2 * x, "square": lambda x: x * x, "neg": lambda x: -x}
toy_rules = {"nonneg": lambda f, p: f(*p) >= 0, "grows": lambda f, p: f(p[0] + 1) > f(*p)}
prep.test(rule_grid, [
    ((toy_opts, toy_rules, [(-1,), (0,), (2,)]),
     {"double": {"nonneg": (-1,), "grows": None},
      "square": {"nonneg": None, "grows": (-1,)},
      "neg": {"nonneg": (2,), "grows": (-1,)}}),
    (({"inv": lambda x: 1 / x}, {"nonneg": lambda f, p: f(*p) >= 0}, [(0,), (1,)]),
     {"inv": {"nonneg": (0,)}}),
])

### EXPLORE: what your test points can and can't do
Run the battery grid. Then a new option, W = 1 − |a − b|/60, is tested twice: once with "ordinary" points only, once with the extremes added. **PREDICT:** does W survive the ordinary points? The extremes?

In [ ]:
def show_grid(grid):
    rules_ = list(next(iter(grid.values())))
    rows = [[opt] + ["ok" if grid[opt][r] is None else f"X at {grid[opt][r]}" for r in rules_] for opt in grid]
    prep.table(rows, header=["option", *rules_])

if prep.ready(rule_grid, {}, {}, []):
    show_grid(rule_grid(battery_options, battery_rules, points))
    W = {"W": lambda a, b: 1 - abs(a - b) / 60}
    print("\nW, ordinary points only:")
    show_grid(rule_grid(W, battery_rules, [(30, 70), (50, 50), (40, 60)]))
    print("\nW, with the extremes added:")
    show_grid(rule_grid(W, battery_rules, [(30, 70), (50, 50), (40, 60), (0, 0), (0, 100)]))

**The aha: test points can only kill.** A finite list of points can show that an option is wrong. It can never show that an option is right, because the next point might kill it. To *accept* an option you need a reason that covers every input. For P: |a − b| is between 0 and 100, so S is between 0 and 1 (R3); |a − b| = |b − a| (R2); S = 1 exactly when |a − b| = 0 (R1).

**Now do practice problem 6.3 on paper** with a grid. For every option you reject, write its counterexample. For every option you accept, write the reason. Then check (letters, e.g. `"B,C"`).

In [ ]:
# Optional, AFTER your paper answer: encode 6.3's options and rules here and run rule_grid
# with test points YOU choose. Did your points find every counterexample you found by hand?

In [ ]:
prep.check("6.3", ...)

---
## 7. How similar are two strings?

The Smart Student Search task builds a similarity score from two standard measures. The exam gives you their definitions, so the skill is running them exactly.

### 7a. Edit distance

**Idea.** The **edit distance** ED(x, y) is the smallest number of single-character edits that turn x into y. An edit is an **insertion**, a **deletion** or a **substitution** (replace one character). For example, ED("cat", "cut") = 1 (substitute a → u).

**How to compute it without guessing.** Line the two words up, character by character, allowing gaps. The **last column** of any lineup is one of three things:
- a letter of x over a gap: that's a **deletion**;
- a gap over a letter of y: that's an **insertion**;
- a letter of x over a letter of y: free if they're equal, otherwise a **substitution** (cost 1).

So if D[i][j] is the edit distance between the first i letters of x and the first j letters of y, then

D[i][j] = min( D[i−1][j] + 1, D[i][j−1] + 1, D[i−1][j−1] + (0 if the i-th letter of x equals the j-th letter of y, else 1) ),

with D[i][0] = i (delete everything) and D[0][j] = j (insert everything). Filling a table of small answers so that each new one reuses earlier ones is called **dynamic programming**.

**PREDICT:** what is ED("arm", "ram")? Guess, then watch the table being filled.

In [ ]:
a, b = "arm", "ram"
D = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
for i in range(len(a) + 1): D[i][0] = i
for j in range(len(b) + 1): D[0][j] = j
log = []
for i in range(1, len(a) + 1):
    for j in range(1, len(b) + 1):
        dele, ins = D[i - 1][j] + 1, D[i][j - 1] + 1
        diag = D[i - 1][j - 1] + (a[i - 1] != b[j - 1])
        D[i][j] = min(dele, ins, diag)
        log.append([a[:i], b[:j], dele, ins, diag, D[i][j]])
prep.table(log, header=["x so far", "y so far", "delete", "insert", "match/sub", "D"])
print()
prep.table([[("" if i == 0 else a[i - 1])] + D[i] for i in range(len(a) + 1)], header=["", "-", *b])

### BUILD 6: `edit_distance`
Write it from the recurrence, without copying the cell above.

In [ ]:
def edit_distance(a, b):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(edit_distance, [
    (("cat", "cut"), 1),
    (("arm", "ram"), 2),
    (("", "abc"), 3),
    (("kitten", "sitting"), 3),
    (("robot", "robot"), 0),
])

### 7b. Jaro similarity, exactly as the Jan 18 paper defines it

The paper calls it "Jaro distance", but it behaves like a **similarity**: 1 means identical.

JD(x, y) = (1/3) · ( m/|x| + m/|y| + (m − t)/m ) if m > 0, and JD = 0 if m = 0, where:
- |x| is the length of x;
- **m** = the number of **matching** characters: characters of x and y that are the same and at most **w = ⌊max(|x|, |y|)/2⌋ − 1** positions apart (⌊ ⌋ means round down);
- **t** = half the number of matching characters that are **out of order**.

**Conventions the definition leaves open (just use these):**
- Go through x from left to right. Match each character with the **first unused** equal character of y inside the window. Each character of y can be matched once.
- To find t, write the matched characters of x in x's order and the matched characters of y in y's order. Count the positions where the two lists differ, and halve it.
- If both strings have length 1, the formula gives w = −1. Use w = 0.

**Smallest example:** MARTHA vs MARHTA. **PREDICT:** work out w, then m, then t, then JD, on paper. Then run.

In [ ]:
def jaro_steps(x, y):
    w = max(0, max(len(x), len(y)) // 2 - 1)
    print(f"x = {x}, y = {y}, window w = floor({max(len(x), len(y))}/2) - 1 = {w}")
    used = [False] * len(y)
    rows, mx = [], []
    for i, ch in enumerate(x):
        lo, hi = max(0, i - w), min(len(y) - 1, i + w)
        found = None
        for j in range(lo, hi + 1):
            if not used[j] and y[j] == ch:
                used[j], found = True, j
                mx.append(ch)
                break
        rows.append([i, ch, f"y[{lo}..{hi}] = {y[lo:hi + 1]}", "-" if found is None else f"y[{found}]"])
    prep.table(rows, header=["i", "x[i]", "window in y", "matched with"])
    my = [y[j] for j in range(len(y)) if used[j]]
    m = len(mx)
    out = sum(p != q for p, q in zip(mx, my))
    print(f"matched, x order: {''.join(mx)}   matched, y order: {''.join(my)}")
    print(f"m = {m}, out-of-order positions = {out}, t = {out / 2}")
    if m:
        print(f"JD = (1/3)({m}/{len(x)} + {m}/{len(y)} + ({m} - {out / 2})/{m}) = {(m / len(x) + m / len(y) + (m - out / 2) / m) / 3:.4f}")

jaro_steps("MARTHA", "MARHTA")

**PREDICT:** CRATE vs TRACE. Both contain the same five letters. Is m = 5? Work out the window first.

In [ ]:
jaro_steps("CRATE", "TRACE")

The window matters: C and T are 3 positions apart, but w = 1, so they don't count as matches. Read the definition every time (move **M1**). Don't trust your sense of "how similar they look".

### BUILD 7: `jaro`
Write it from the definition and the conventions above. Return a float.

In [ ]:
def jaro(x, y):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(jaro, [
    (("MARTHA", "MARHTA"), 17 / 18),
    (("CRATE", "TRACE"), 11 / 15),
    (("DIXON", "DICKSONX"), 23 / 30),
    (("abc", "xyz"), 0.0),
    (("abc", "abc"), 1.0),
])

### EXPLORE: the properties you'll need later
Smart Student Search turns on three facts about JD and ED. **PREDICT** each before running:
1. Is JD(x, y) always between 0 and 1?
2. Is JD(x, y) = 1 exactly when x = y? Is ED(x, y) = 0 exactly when x = y?
3. Are JD and ED symmetric: JD(x, y) = JD(y, x), and ED(x, y) = ED(y, x)?

In [ ]:
if prep.ready(jaro, "a", "a") and prep.ready(edit_distance, "a", "a"):
    rng = random.Random(0)
    words = ["".join(rng.choice("abr") for _ in range(rng.randint(1, 7))) for _ in range(400)]
    pairs = [(rng.choice(words), rng.choice(words)) for _ in range(5000)]
    jd = [jaro(x, y) for x, y in pairs]
    print("JD range seen:", round(min(jd), 4), "to", round(max(jd), 4))
    print("pairs with JD = 1 but x != y:", sum(abs(v - 1) < 1e-12 and x != y for v, (x, y) in zip(jd, pairs)))
    print("pairs with ED = 0 but x != y:", sum(edit_distance(x, y) == 0 and x != y for x, y in pairs))
    print("JD asymmetric pairs:", sum(abs(jaro(x, y) - jaro(y, x)) > 1e-12 for x, y in pairs))
    print("ED asymmetric pairs:", sum(edit_distance(x, y) != edit_distance(y, x) for x, y in pairs))

Random tests only *suggest* these facts (section 6: points can only kill). Here are the reasons:
- **Range:** m ≤ |x|, m ≤ |y| and m − t ≤ m, so each of the three fractions is between 0 and 1, and so is their average.
- **JD = 1 exactly when x = y:** all three fractions must equal 1, so every character matches (m = |x| = |y|) and none is out of order (t = 0).
- **Symmetry:** an edit from x to y can be undone by the reverse edit from y to x, so ED is symmetric. For JD, swapping x and y swaps the first two fractions and doesn't change m or t.

*Note:* the greedy matching convention for m could, in principle, depend on which string you scan. It held on every random test above. For the exam, treat JD as symmetric.

---
## 8. Decoding an unknown language

**Idea.** If an English word appears in several sentences, its translation should appear in all of their translations. So:
1. **Align:** for one English word, put every sentence containing it side by side and look for what their translations share.
2. **Subtract:** remove symbols you've already explained by other words.
3. **Order:** label each symbol by its job in the sentence, and look for the pattern.
4. **Test every rule on every sentence,** and actively look for a second rule that fits just as well.

**Smallest example.** An invented robot-arm command language:

In [ ]:
TOY = [
    ("Lift the box.",      ["dax", "mok"]),
    ("Lift the big ball.", ["dax", "pelo"]),
    ("Drop the big box.",  ["fen", "moko"]),
    ("Drop the ball.",     ["fen", "pel"]),
]
for word in ["lift", "drop", "box", "ball", "big"]:
    hits = [(k + 1, words) for k, (eng, words) in enumerate(TOY) if word in eng.lower()]
    common = set.intersection(*(set(w) for _, w in hits))
    print(f"{word:>5}: sentences {[k for k, _ in hits]}  translations {[' '.join(w) for _, w in hits]}  shared: {common or 'nothing'}")

**PREDICT:** "lift" and "drop" align perfectly. Why do "box", "ball" and "big" share nothing? What should you look at instead of whole words?

<details><summary>Check your reasoning</summary>

The words change shape: "box" is *mok* in one sentence and *moko* in another. Whole words don't repeat, but a **piece** does: *mok* is inside both, *pel* is inside *pelo* and *pel*, and the extra *-o* appears exactly when "big" does. When whole-word alignment fails, align pieces: a shared start (a **prefix**) or a shared end (a **suffix**).
</details>

### Try to break your own hypothesis
Suppose you only had sentences (1) and (2). Then *-o* shows up in "the big ball" and nowhere else. Two hypotheses fit equally well: **H1** "*-o* means big" and **H2** "*-o* means the thing is a ball". The cell adds one sentence at a time and tests both.

In [ ]:
order = [2, 1, 3, 4]
alive = {"H1: -o means big", "H2: -o means ball"}
rows = []
for k in order:
    eng, words = TOY[k - 1]
    has_o = any(w.endswith("o") for w in words)
    big, ball = "big" in eng, "ball" in eng
    if has_o != big: alive.discard("H1: -o means big")
    if has_o != ball: alive.discard("H2: -o means ball")
    rows.append([f"({k}) {eng}", " ".join(words), has_o, big, ball, "; ".join(sorted(alive))])
prep.table(rows, header=["sentence", "translation", "has -o", "big?", "ball?", "still alive"])

**The aha: when two features always appear together, no amount of staring can separate them. You need a sentence where they come apart.** Sentence (3) has "big" without "ball", and that kills H2. So for every rule you find, ask: *what else was always present when this piece appeared?* Then look for the sentence that separates them. If there isn't one, say so.

### BUILD 8: `shared_symbols`
`pairs` is a list of `(english_sentence, list_of_symbols)`. Return the **set** of symbols that appear in the translation of *every* sentence containing `word`. A sentence contains the word if one of its English words, lower-cased and stripped of punctuation, **starts with** it (so "apple" also finds "apples"). Return an empty set if no sentence contains the word.

In [ ]:
def shared_symbols(pairs, word):
    # your code here
    raise NotImplementedError

In [ ]:
arm_pairs = [("The arm moves.", ["lumo", "dax"]), ("The arm stops.", ["lumo", "fen"]), ("The wheel moves.", ["kiro", "dax"])]
prep.test(shared_symbols, [
    ((arm_pairs, "arm"), {"lumo"}),
    ((arm_pairs, "move"), {"dax"}),
    ((arm_pairs, "wheel"), {"kiro", "dax"}),     # only one sentence, so it can't tell kiro from dax yet
    ((arm_pairs, "gripper"), set()),
])

### Practice problem 6.5
Words you need for 6.5: the **subject** is who does the action ("the girl" in "The girl saw the robot"); the **object** is who or what it's done to ("the robot"). **Plural** means more than one. **Tense** is when the action happens (present "sees", past "saw").

**PREDICT:** run `shared_symbols` on the five 6.5 sentences for the word "robot", treating whole words as symbols. What will it return?

In [ ]:
S65 = [
    ("The robot builds the car.", "robo veln mor"),
    ("The girl saw the robot.",   "mina robon taki"),
    ("The robots see the girl.",  "karobo minan tak"),
    ("The girl built the cars.",  "mina kaveln mori"),
    ("The robot saw the girl.",   "robo minan taki"),
]
if prep.ready(shared_symbols, [], "x"):
    print(shared_symbols([(e, r.split()) for e, r in S65], "robot"))

Same situation as the toy: the words change shape. **Now decode 6.5 on paper.** Make a table with one row per sentence and columns for each word and its job. For each piece you explain (a prefix, a suffix, the word order), write down another feature that *also* fits it in some sentences, and find the sentence that separates the two.

### BUILD 9: `encode`, your grammar as code
Once you've decoded the rules on paper, write them as a function that takes an English sentence of the form "The X verb the Y." and returns the robot-language sentence. **The tests are the five given sentences**: this is "test your hypothesis on every sentence", done by machine.

In [ ]:
def encode(english):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(encode, [((e,), r) for e, r in S65])

When all five pass, answer (a) and (b). Use your rules on paper first. You may call your `encode` as a final check.

In [ ]:
prep.check("6.5a", ...)   # option letter
prep.check("6.5b", ...)   # the robot-language sentence, as a string

---
## 9. LLM sense: tokens, not letters

### 9a. What the model actually sees

**Idea.** A **large language model (LLM)** never sees letters. Its input is first chopped into **tokens**: chunks of text from a fixed list called the **vocabulary**. Each token is replaced by a number, its **token ID**. The model only ever receives the list of IDs. For example, with a vocabulary containing "straw" and "berry", the word "strawberry" becomes two tokens.

A simple way to chop: at each position, take the **longest** vocabulary piece that matches the text there (this is called greedy longest match). If nothing matches, take a single character.

**PREDICT:** with the vocabulary below, how will "strawberry" be chopped? Write the tokens down, then run.

In [ ]:
VOCAB = ["straw", "berry", "st", "raw", "ber", "ry", "s", "t", "r", "a", "w", "b", "e", "y", " "]
IDS = {tok: 1000 + 37 * k for k, tok in enumerate(VOCAB)}

text, i, rows = "strawberry", 0, []
while i < len(text):
    options = [v for v in VOCAB if text.startswith(v, i)]
    piece = max(options, key=len) if options else text[i]
    rows.append([i, text[i:], ", ".join(options), piece, IDS.get(piece, "?")])
    i += len(piece)
prep.table(rows, header=["position", "text left", "vocab pieces that match", "chosen", "token ID"])

### BUILD 10: `greedy_tokenize`
Return the list of tokens (strings) produced by greedy longest match.

In [ ]:
def greedy_tokenize(text, vocab):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(greedy_tokenize, [
    (("strawberry", VOCAB), ["straw", "berry"]),
    (("robot", ["ro", "bot", "r", "o", "b", "t"]), ["ro", "bot"]),
    (("xyz", ["ab"]), ["x", "y", "z"]),
    (("aaa", ["a", "aa"]), ["aa", "a"]),
])

**PREDICT:** the model receives only the IDs printed below. Using *only those numbers*, how many r's are in the word? What would the model need to have memorised to answer?

In [ ]:
if prep.ready(greedy_tokenize, "a", ["a"]):
    toks = greedy_tokenize("strawberry", VOCAB)
    print("what you see:   ", toks)
    print("what the model gets:", [IDS[t] for t in toks])

**The aha: an LLM sees tokens, not letters.** The letters inside a token are invisible to it. It can only answer "how many r's?" if, during training, it happened to learn the spelling of each token, and then it still has to count, which it does unreliably. Tasks that need **character-level detail** (counting letters or characters, reversing a word, exact lengths) are where LLMs fail confidently.

**EXPLORE. PREDICT:** what happens if you write the word with spaces, "s t r a w b e r r y"? Could the model count now?

In [ ]:
if prep.ready(greedy_tokenize, "a", ["a"]):
    toks = greedy_tokenize("s t r a w b e r r y", VOCAB)
    print([t for t in toks if t != " "])

### 9b. The training cutoff: why it makes things up

**Idea.** An LLM learns from text collected up to some date, its **training cutoff**. After that its knowledge is frozen. Offline, with no internet or tools, it can't look anything up. And it *always* produces some next token: it picks from a probability distribution over tokens it knows, and that distribution always adds up to 1. A confident, plausible, made-up answer is called a **hallucination**.

**Smallest example.** A toy "model" that predicts the next word from the two words before it, trained on a few old match reports.

**PREDICT:** you ask it to complete "yesterday india scored". What will it say? Can it say "I don't know"?

In [ ]:
corpus = [
    "india scored 287 runs", "india scored 312 runs", "india scored 287 runs",
    "australia scored 250 runs", "india won the match",
]
next_word = {}
for sentence in corpus:
    w = sentence.split()
    for k in range(len(w) - 2):
        next_word.setdefault((w[k], w[k + 1]), Counter())[w[k + 2]] += 1

prompt = "yesterday india scored".split()
context = tuple(prompt[-2:])
dist = next_word[context]
total = sum(dist.values())
prep.table([[word, c, round(c / total, 3)] for word, c in dist.most_common()], header=["next word", "count", "probability"])
print("model's answer: yesterday india scored", dist.most_common(1)[0][0], "...")

The toy model never even looked at "yesterday". It answered from old data with probability 0.67, because "I don't know" was not in its training text. Real LLMs are much bigger, and fine-tuning teaches them to admit uncertainty sometimes, but the core problem stays: **facts after the cutoff, or facts that aren't in the prompt, come out as plausible guesses.**

### Three questions for any LLM task
1. **Does it need character-level work?** (counting letters or characters, exact lengths) → likely wrong.
2. **Does it need facts that are neither in the prompt nor in common training text?** (yesterday's events, who published an unlabelled headline, when it was written) → likely a confident guess.
3. Otherwise, is it **pattern matching on text it's given or very common text?** (tone, topic, sentiment, language, summarising, translating common phrases, spotting quotation marks) → usually fine.

**Now do practice problem 6.7 on paper**, putting each task into one of the three buckets, then check (letters, e.g. `"B,C"`).

In [ ]:
prep.check("6.7", ...)

---
## 10. Exam walkthroughs: Jan 18, Section A1

> Do these **after** taking the Jan 18 paper as your diagnostic. Open `papers/2026-stage1-jan18/paper.pdf`. Each task has: the question, a cell for your attempt, a hint ladder, then the check.

### 10.1 Cracking the Imperial Glyphs (page 1; options on page 2)

Naruto must say **"The tree makes the palace beautiful."** in imperial glyphs. The six known sentences are in the cell below. The options (from page 2) are:

| | | | |
|---|---|---|---|
| **A.** 樹造宮美 | **B.** 宮造樹美 | **C.** 樹美造宮 | **D.** 美樹造宮 |
| **E.** 樹造美宮 | **F.** 宮美造樹 | | |

**Step 1: which words do you need?** You need glyphs for *tree*, *make(s)*, *palace* and *beautiful*, plus the word order. Fill in your dictionary by aligning sentences by hand (section 8).

In [ ]:
GLYPHS = [
    ("This is a beautiful apple.",               list("這美蘋果")),
    ("This is an apple on a tree.",              list("這蘋果樹")),
    ("They serve green apples in the palace.",   list("服綠蘋果宮")),
    ("Does the tree make the apple beautiful?",  list("樹造蘋果美？")),
    ("There is a palace on the black mountain.", list("這黑山宮")),
    ("The mountain paints the palace black.",    list("山塗宮黑")),
]
my_dictionary = {"apple": ..., "tree": ..., "make": ..., "palace": ..., "beautiful": ...}   # glyph(s) for each word

<details><summary>Hint 1</summary>

Start with the word that appears most often: "apple" is in four sentences. Which glyphs are in all four translations? Then "tree" is in two sentences. Which glyphs do they share, once you remove the apple glyphs?
</details>

<details><summary>Hint 2</summary>

"Make" appears in only one sentence, so alignment alone gives you that whole sentence. **Subtract** every glyph you've already explained (tree, apple, beautiful, the question mark). What's left?
</details>

**Step 2: check your dictionary is consistent.** For each word, the cell checks that your glyph appears in every sentence containing that word. This is necessary but not sufficient: the apple glyphs also appear in every "tree" sentence, which is why step 2 of the method is *subtract*.

In [ ]:
if prep.ready(shared_symbols, [], "x"):
    for word, g in my_dictionary.items():
        if g is ...:
            print(f"{word}: fill in your glyph(s) first."); continue
        shared = shared_symbols(GLYPHS, word)
        ok = all(ch in shared for ch in g)
        print(f"{word}: {'consistent: in every sentence with this word' if ok else 'NOT consistent: some sentence with this word lacks it'}")

**Step 3: the word order.** Two sentences have the same shape as the target: "Does the tree make the apple beautiful?" and "The mountain paints the palace black." Label each glyph by its job: who does it, the action, the thing acted on, the result.

<details><summary>Hint 3</summary>

Compare "This is a **beautiful apple**" with "make **the apple beautiful**". Where does 美 sit relative to 蘋果 in each? An adjective that *describes* a noun and an adjective that is the *result* of an action are in different places. Which job does "beautiful" have in the target sentence?
</details>

<details><summary>Hint 4</summary>

In both model sentences the order is: doer, action, thing acted on, result. Build the target in that order and compare it with the options. Exactly one matches, and option E is the trap for anyone who copied the "beautiful apple" order.
</details>

*Side note (section 8's aha):* "mountain" and "black" appear in exactly the same two sentences, so alignment alone can't tell 山 from 黑. The word order you found separates them. You don't need them for this question.

In [ ]:
prep.check("J18-glyphs", ...)   # option letter

### 10.2 Filtering Fakes with Faulty Oracle (pages 2–3)

A small **offline** multimodal model (it reads images as well as text, and has no internet) extracts features from a headline and its image:

| ID | Feature | Prompt |
|---|---|---|
| F1 | Character count | "Count the number of characters in the headline." |
| F2 | Estimated publication year | "Guess the publication year of this headline." |
| F3 | Tone | "Classify the headline as sensational or neutral." |
| F4 | Image–headline match score | "Rate from 1–5 how well the image supports the headline." |
| F5 | Language identification | "Identify the language ... return the ISO-639-1 two-letter code" (a standard code such as en for English, hi for Hindi) |
| F6 | Named outlet prediction | "Which news outlet likely published this headline?" |
| F7 | Topic classification | "Classify topic: science, sports, politics, ..." |
| F8 | Quote/attribution presence | "Does the headline contain a quote or attribution? (yes/no)" |

Which subset is **structurally** the most unreliable and should be excluded?

**Step 1: sort every feature into one of section 9's three buckets.**

In [ ]:
my_buckets = {
    "needs counting characters": [...],
    "needs world knowledge it doesn't have": [...],
    "pattern matching it can do": [...],
}

<details><summary>Hint 1</summary>

For each feature ask two questions: is everything it needs **in the input** (the headline and the image)? Does it need **character-level counting**?
</details>

<details><summary>Hint 2</summary>

One feature is the strawberry problem from section 9 in disguise.
</details>

<details><summary>Hint 3</summary>

A headline doesn't print its own date or its publisher's name. What would the model need to *know* to answer F2 or F6, and does a small offline model have it? Compare with F5 or F8, where everything needed is right there on the page.
</details>

**Step 2: check your sorting and find which option it is.** The cell checks that every feature is in exactly one bucket, then tells you which option letter equals your "exclude" set (the first two buckets combined).

In [ ]:
ORACLE_OPTIONS = {"A": {"F3", "F4", "F7"}, "B": {"F1", "F2", "F6"}, "C": {"F1", "F2", "F8"},
                  "D": {"F2", "F3", "F7"}, "E": {"F3", "F4", "F6"}, "F": {"F4", "F5", "F7"}}
placed = [f for fs in my_buckets.values() for f in fs]
if any(f is ... for f in placed):
    print("Fill in your buckets first (replace the ...).")
else:
    missing = sorted({f"F{k}" for k in range(1, 9)} - set(placed))
    repeated = sorted({f for f in placed if placed.count(f) > 1})
    if missing or repeated:
        print("missing:", missing, " in two buckets:", repeated)
    else:
        exclude = set(my_buckets["needs counting characters"]) | set(my_buckets["needs world knowledge it doesn't have"])
        match = [k for k, v in ORACLE_OPTIONS.items() if v == exclude]
        print(f"your exclude set {sorted(exclude)} is option {match[0]}" if match else
              f"your exclude set {sorted(exclude)} matches no option: rethink the borderline features")

In [ ]:
prep.check("J18-oracle", ...)   # option letter

### 10.3 Smart Student Search (pages 3–4)

Student IDs are s = (c, f, l) = (section, first name, last name). Sim(s, t) must satisfy:
1. Sim = 1 **if and only if** all fields match exactly.
2. Sim = 0 if the section codes differ.
3. Sim(s, t) = Sim(t, s) (symmetric).
4. When sections match, first-name mismatches reduce the score **more** than last-name mismatches.
5. 0 ≤ Sim ≤ 1 always.

𝟙[c = c′] is the **indicator**: 1 if the sections are equal, 0 otherwise. The options are coded in the cell below, exactly as printed on page 4.

**Step 1: read the definition (M1). Compute JD("ARYAN", "ARJUN") by hand,** with the exam's formula and section 7's conventions. Write w, m and t.

In [ ]:
my_w, my_m, my_t = ..., ..., ...

if any(v is ... for v in (my_w, my_m, my_t)):
    print("Fill in w, m and t first.")
elif prep.ready(jaro, "a", "a"):
    print("w:", "correct" if my_w == max(0, 5 // 2 - 1) else "re-read the window definition")
    mine = (my_m / 5 + my_m / 5 + (my_m - my_t) / my_m) / 3 if my_m else 0.0
    print("your m and t give the same JD as your jaro():", abs(mine - jaro("ARYAN", "ARJUN")) < 1e-9)

**Step 2: what do you know about the building blocks?** From section 7: is JD always in [0, 1]? Is it 1 only for equal strings? Is it symmetric? Same questions for e^(−ED). And what is JD of two *different* section codes, such as "9C" and "9D"? Compute it by hand.

**Step 3: the grid.** On paper, draw 6 options × 5 rules. For each option, hunt for **one** counterexample, using section 6's hiding places:
- rule 2: two IDs with different sections;
- rules 1 and 5: two identical IDs (is Sim exactly 1? can it exceed 1?);
- rule 3: swap s and t;
- rule 4: take a pair that differs only in the first name, then move the same mismatch into the last name. Rule 4 says the score must go **up**.

<details><summary>Hint 1</summary>

Look at the factor in front of each option. Which option doesn't use the indicator for the section? What does its factor give for "9C" vs "9D"?
</details>

<details><summary>Hint 2</summary>

For rule 4, find the **weight** on the first-name term and on the last-name term in each option. The first name must carry the bigger penalty. Which options give the same weight to both, or the bigger weight to the last name?
</details>

<details><summary>Hint 3</summary>

One option's weights use |f′| and |l′|, the lengths of the fields of t only. What happens to those weights when you swap s and t? Try s = (9C, Al, Shah) and t = (9C, Alan, Shah).
</details>

**Step 4: test your counterexamples.** For each option you reject, give the rule number and the pair (s, t) that breaks it. Use `None` for an option you think survives everything. The cell checks that each counterexample *really* breaks its rule, and never says which option is right. For rule 4, give a pair that differs only in the first name.

In [ ]:
def ind(x, y):
    return 1 if x == y else 0

SIM = {
    "A": lambda s, t: ind(s[0], t[0]) * (jaro(s[1], t[1]) + jaro(s[2], t[2])) / 2,
    "B": lambda s, t: jaro(s[0], t[0]) * math.exp(-(1 - jaro(s[1], t[1])) - 2 * (1 - jaro(s[2], t[2]))),
    "C": lambda s, t: ind(s[0], t[0]) * math.exp(-2 * (1 - jaro(s[1], t[1])) - (1 - jaro(s[2], t[2]))),
    "D": lambda s, t: ind(s[0], t[0]) * math.exp(-edit_distance(s[1], t[1])) * math.exp(-2 * edit_distance(s[2], t[2])),
    "E": lambda s, t: ind(s[0], t[0]) * (2 * len(t[1]) * jaro(s[1], t[1]) + len(t[2]) * jaro(s[2], t[2])) / (2 * len(t[1]) + len(t[2])),
    "F": lambda s, t: ind(s[0], t[0]) * (2 * jaro(s[1], t[1]) + jaro(s[2], t[2])),
}

def check_kill(opt, rule, s, t):
    f = SIM[opt]
    v = f(s, t)
    if rule == 1:
        broken = (s == t) != math.isclose(v, 1, abs_tol=1e-12)
        why = f"Sim = {v:.4f} and the IDs are {'equal' if s == t else 'different'}"
    elif rule == 2:
        if s[0] == t[0]:
            return f"{opt}: the sections match, so this pair can't test rule 2."
        broken, why = v != 0, f"sections differ and Sim = {v:.4f}"
    elif rule == 3:
        v2 = f(t, s)
        broken, why = not math.isclose(v, v2, abs_tol=1e-12), f"Sim(s,t) = {v:.4f}, Sim(t,s) = {v2:.4f}"
    elif rule == 4:
        if not (s[0] == t[0] and s[2] == t[2] and s[1] != t[1]):
            return f"{opt}: for rule 4, give a pair with the same section and last name and different first names."
        v_last = f((s[0], s[2], s[1]), (t[0], t[2], t[1]))
        broken = v >= v_last
        why = f"first-name mismatch gives {v:.4f}, the same mismatch in the last name gives {v_last:.4f}"
    elif rule == 5:
        broken, why = not (0 <= v <= 1), f"Sim = {v:.4f}"
    else:
        return f"{opt}: rule should be 1-5."
    return f"{opt}: {'breaks' if broken else 'does NOT break'} rule {rule} ({why})."

my_kills = {   # option: (rule, s, t), or None if you think the option survives every rule
    "A": ...,
    "B": ...,
    "C": ...,
    "D": ...,
    "E": ...,
    "F": ...,
}

if prep.ready(jaro, "a", "a") and prep.ready(edit_distance, "a", "a"):
    for opt, claim in my_kills.items():
        if claim is ...:
            print(f"{opt}: fill in a counterexample or None.")
        elif claim is None:
            print(f"{opt}: you say it survives. Can you give a reason for each rule that covers every input?")
        else:
            print(check_kill(opt, *claim))

Exactly one option should be left standing. Remember section 6: for that one, your test points prove nothing. Write a one-line reason for each of the five rules.

In [ ]:
prep.check("J18-search", ...)   # option letter

### 10.4 Pikachu's Path to Victory (pages 5–6)

Pikachu moves like a knight on a 5 × 5 board, from (0, 0) to (4, 4), and never lands on a pad twice. What is the maximum number of pads, counting both ends? Options: 15, 19, 20, 21, 25, 28.

**Step 1: the colouring.** Does a knight move change the colour? (Section 1.) What colours are (0, 0) and (4, 4)?

**Step 2: the ceiling.** Count the dark and light cells on 5 × 5. If the path alternates colours and starts and ends on the same colour, what does that tell you about (a) whether the number of pads is odd or even, and (b) the most it can be? **Which options does this kill without any search?**

<details><summary>Hint 1</summary>

Same colour at both ends means the path looks like D, L, D, ..., L, D: one more D than L. So the number of pads is odd.
</details>

<details><summary>Hint 2</summary>

How many D cells are there, and how many L cells? The path can't use more D cells than exist, and it needs exactly one fewer L than D.
</details>

**Step 3: is the ceiling reached?** Section 2 showed that a ceiling can be loose: on 3 × 3 the knight couldn't get anywhere near it. On paper, try to build a long path, visiting the awkward corner and edge pads early, before they get cut off. Then let the search confirm it.

In [ ]:
my_bound = ...   # your colour ceiling for Pikachu

if my_bound is ...:
    print("Fill in your ceiling first.")
elif prep.ready(longest_path, 2, (0, 0), (0, 1), ORTH):
    found = longest_path(5, (0, 0), (4, 4), KNIGHT)
    if found == my_bound:
        print("The search reaches your ceiling exactly, so the ceiling is the answer.")
    elif found < my_bound:
        print("The search never reaches your ceiling. Either it is loose (as on 3x3) or your counting is off. Recheck step 2.")
    else:
        print("The search found a path longer than your ceiling allows, so your ceiling is wrong. Recheck step 2.")

On the exam you can't run the search. A clean ceiling that also appears among the options is *often* the intended answer, but "often" isn't proof (remember 3 × 3). Use the colouring to cut the options, then spend what time you can afford trying to build a path.

In [ ]:
prep.check("J18-pikachu", ...)   # option letter or number of pads

---
## 11. Connect

Answer in your own words in the cell below. Then copy the best sentence into `log/ERROR_LOG.md` under "Moves I own".

1. **Invariants and telescoping.** In 6.1 the colour flips on every move, and in 6.6 (section 5's explore) the bit flips at every change. In Week 4 (problem 4.5), the [1, −1] differences of a signal add up to last − first. Write one sentence explaining why, in all three, *only the two ends* decide the parity or the total.
2. **Order-only.** Option C of Smart Search wraps a weighted sum inside exp, and exp is monotone increasing (if a > b then eᵃ > eᵇ). If Sim were used only to *rank* search results, would removing the exp change the ranking? Which of the five rules would then break? Compare with AUC (Week 1), which ignores any monotone change of the scores, and Precision@k (Week 4).
3. **Does the ceiling bind?** The 3 × 3 knight board had a ceiling that was never reached. In Set 05 (problem 5.2) a budget of 10 GPU-hours was never fully used. What do these have in common, and what extra check do both need before you trust the number?

*Your answers here.*